# API Latency and SLAs — CDF and Percentiles

An ML inference API serves predictions. Response latency $X$ (in milliseconds) follows a **log-normal distribution** — a common model for latency because response times are always positive and right-skewed (most requests are fast, a few are very slow).

$$
\ln X \sim N(\mu_{\ln}, \sigma_{\ln})
$$

The service has a **Service Level Agreement (SLA)**: 95% of requests must complete within 200 ms.

Questions:

1. What does the latency distribution look like?
2. What are the **P50, P95, P99** latency values?
3. Is the current SLA being met? What fraction of requests finish within 200 ms?
4. A slow model is added to the pipeline. How does this affect the P99?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# Log-normal parameters calibrated so median ~ 80ms, P99 ~ 180ms
mu_ln = np.log(80)
sigma_ln = 0.6

n_requests = 100_000
latency = rng.lognormal(mean=mu_ln, sigma=sigma_ln, size=n_requests)

## Step 1: Plot the latency distribution

The PDF shows that most requests are fast, with a long right tail of slow requests.
The tail is where SLA violations happen.

In [ ]:
plt.figure(figsize=(10, 4))
plt.hist(latency, bins=150, density=True, alpha=0.7)
plt.axvline(200, linestyle="--", color="red", label="SLA threshold (200 ms)")
plt.xlim(0, 600)
plt.xlabel("Latency (ms)")
plt.ylabel("Density")
plt.title("ML API latency distribution (log-normal)")
plt.legend()
plt.show()

print(f"Mean latency:   {latency.mean():.1f} ms")
print(f"Median latency: {np.median(latency):.1f} ms")

## Step 2: Percentiles are the inverse CDF

In production systems, latency is reported as **percentiles**:

- **P50** — $F^{-1}(0.50)$ — the median: half of requests are faster than this.
- **P95** — $F^{-1}(0.95)$ — 95% of requests are faster than this.
- **P99** — $F^{-1}(0.99)$ — the "tail latency": 99% of requests are faster.

P99 matters most for user experience: a user making 100 requests will likely experience the P99 at least once.

In [ ]:
percentiles = [50, 75, 90, 95, 99, 99.9]
values = np.percentile(latency, percentiles)

for p, v in zip(percentiles, values):
    print(f"P{p:<5} = {v:6.1f} ms")

# Plot the empirical CDF with percentile markers
sorted_latency = np.sort(latency)
ecdf = np.arange(1, n_requests + 1) / n_requests

plt.figure(figsize=(10, 5))
plt.plot(sorted_latency, ecdf)
for p, v in zip([50, 95, 99], np.percentile(latency, [50, 95, 99])):
    plt.axvline(v, linestyle="--", alpha=0.7, label=f"P{p} = {v:.0f} ms")
plt.axvline(200, linestyle="-", color="red", linewidth=2, label="SLA = 200 ms")
plt.xlim(0, 600)
plt.xlabel("Latency (ms)")
plt.ylabel("F(x) = P(X ≤ x)")
plt.title("Empirical CDF of API latency")
plt.legend()
plt.show()

## Step 3: Is the SLA being met?

The SLA requires $P(X \le 200) \ge 0.95$.

This is exactly $F(200)$ — reading the CDF at 200 ms.

In [ ]:
sla_threshold = 200
sla_target = 0.95

sla_actual = np.mean(latency <= sla_threshold)
print(f"P(X ≤ {sla_threshold} ms) = {sla_actual:.4f}  ({sla_actual:.1%})")
print(f"SLA target:               {sla_target:.1%}")
print()
if sla_actual >= sla_target:
    print("✓ SLA is being met.")
else:
    shortfall = sla_target - sla_actual
    print(f"✗ SLA is NOT met. Shortfall: {shortfall:.1%} of requests are violating.")

## Step 4: Adding a slow model to the pipeline

Suppose a new, heavier model is added that adds extra latency drawn from a second log-normal.
Total latency = fast model + slow model.

How does this shift the CDF and affect P99?

In [ ]:
# Slow model adds latency: log-normal with median ~40ms
extra_latency = rng.lognormal(mean=np.log(40), sigma=0.8, size=n_requests)
latency_combined = latency + extra_latency

for label, data in [("Original", latency), ("+ Slow model", latency_combined)]:
    p50, p95, p99 = np.percentile(data, [50, 95, 99])
    sla = np.mean(data <= sla_threshold)
    print(f"{label:15s}  P50={p50:6.1f}ms  P95={p95:6.1f}ms  P99={p99:6.1f}ms  SLA={sla:.1%}")

# Plot both CDFs
sorted_combined = np.sort(latency_combined)
plt.figure(figsize=(10, 5))
plt.plot(sorted_latency, ecdf, label="Original")
plt.plot(sorted_combined, ecdf, label="+ Slow model")
plt.axvline(sla_threshold, linestyle="--", color="red", label=f"SLA = {sla_threshold} ms")
plt.axhline(sla_target, linestyle=":", color="gray", label=f"SLA target = {sla_target:.0%}")
plt.xlim(0, 800)
plt.xlabel("Latency (ms)")
plt.ylabel("F(x) = P(X ≤ x)")
plt.title("CDF shift after adding slow model")
plt.legend()
plt.show()

## Observation

- **Percentiles are the inverse CDF** $F^{-1}(p)$. P50, P95, P99 are the standard language of production ML systems and appear in every monitoring dashboard.
- **SLA compliance is $F(\text{threshold})$** — the probability that a request finishes within the threshold. Checking SLA compliance is literally reading the CDF at one point.
- P99 is the most operationally important percentile: it captures the experience of users who make many requests, and it is highly sensitive to the tail of the distribution.
- Adding a slow model **shifts the CDF left** (toward higher latency), reducing $F(200)$ and potentially breaching the SLA even if the mean latency only increases modestly.
- The mean latency $E[X]$ can look fine while the P99 is severely degraded — this is why production systems monitor percentiles, not just averages.